In [2]:
import threading
from bleak import BleakClient
import asyncio
import time

class BleakClientThread:
    """
    A class to manage asynchronous BLE operations using Bleak and threading.
    see ref at https://github.com/hbldh/bleak

    Attributes:
        address (str): The BLE device address.
        loop (asyncio.new_event_loop): The event loop for asynchronous operations.
        client (BleakClient): The Bleak BLE client.
        thread (threading.Thread): The thread running the event loop.
        connected (bool): Connection status of the BLE client.

    Methods:
        run_loop(loop): Runs the given event loop.
        connect(): Asynchronously connects to the BLE device.
        disconnect(): Asynchronously disconnects from the BLE device.
        read_characteristic(char_uuid): Reads data from a specified characteristic.
        write_characteristic(char_uuid, data): Writes data to a specified characteristic.
        start(): Starts the thread and the event loop.
        stop(): Stops the event loop and joins the thread.
        perform_action(coro): Schedules a coroutine to be run in the event loop thread.
        prot_open(): Placeholder for opening protocol.
        prot_close(): Placeholder for closing protocol.
        prot_stop(): Placeholder for stopping protocol.
    """
    def __init__(self, address):
        """
        Initializes the BleakClientThread with a given BLE device address.

        Parameters:
            address (str): The BLE device address.
        """

        self.address = address
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)  # Set the event loop for this thread.
        self.client = BleakClient(address, loop=self.loop)  # Ensure BleakClient uses this loop.
        self.thread = threading.Thread(target=self.loop.run_forever)
        self.thread.daemon = True
        self.connected = False

    def run_loop(self, loop):
        """
        Runs the given event loop.

        Parameters:
            loop (asyncio.new_event_loop): The event loop to run.
        """
        asyncio.set_event_loop(loop)
        loop.run_forever()

    async def connect(self):
        """
        Asynchronously connects to the BLE device.

        Returns:
            bool: True if the connection was successful, False otherwise.
        """
        self.connected = await self.client.connect()
        return self.connected

    async def disconnect(self):
        """
        Asynchronously disconnects from the BLE device.
        """
        await self.client.disconnect()
        self.connected = False

    async def read_characteristic(self, char_uuid):
        """
        Reads data from a specified characteristic.

        Parameters:
            char_uuid (str): The UUID of the characteristic to read from.

        Returns:
            The data read from the characteristic.
        """
        return await self.client.read_gatt_char(char_uuid)

    async def write_characteristic(self, char_uuid, data):
        """
        Writes data to a specified characteristic.

        Parameters:
            char_uuid (str): The UUID of the characteristic to write to.
            data: The data to write.
        """
        await self.client.write_gatt_char(char_uuid, data)

    def start(self):
        """
        Starts the thread and the event loop.
        """
        self.thread.start()

    def stop(self):
        """
        Stops the event loop and joins the thread.
        """
        self.loop.call_soon_threadsafe(self.loop.stop)
        self.thread.join()

    def perform_action(self, coro):
        """
        Schedules a coroutine to be run in the event loop thread.

        Parameters:
            coro: The coroutine to schedule.

        Returns:
            The result of the coroutine, or None if an error occurred.
        """
        fut = asyncio.run_coroutine_threadsafe(coro, self.loop)
        try:
            result = fut.result()
            return result
        except Exception as e:
            print(f"An error occurred: {e}")
            return None

    def prot_open(self):
        """
        Placeholder for opening protocol. Not implemented yet.
        """
        raise NotImplementedError("prot_open function not implemented yet")

    def prot_close(self):
        """
        Placeholder for closing protocol. Not implemented yet.
        """
        raise NotImplementedError("prot_close function not implemented yet")

    def prot_stop(self):
        """
        Placeholder for stopping protocol. Not implemented yet.
        """
        raise NotImplementedError("prot_stop function not implemented yet")




class IndyClientThread(BleakClientThread):
    """
    A class to manage specific BLE operations for Indy devices using the BleakClientThread as a base.

    Attributes:
        OPEN_MOTOR_HDLE (str): UUID for opening the motor characteristic.
        CLOSE_MOTOR_HDLE (str): UUID for closing the motor characteristic.
        data_go (bytearray): Data to send to initiate an action.
        data_nogo (bytearray): Data to send to stop an action.
    """
    def __init__(self, address):
        """
        Initializes the IndyClientThread with a given BLE device address.

        Parameters:
            address (str): The BLE device address.
        """
        super().__init__(address)
        self.OPEN_MOTOR_HDLE = "0000fe48-8e22-4541-9d4c-21edae82ed19"
        self.CLOSE_MOTOR_HDLE = "0000fe49-8e22-4541-9d4c-21edae82ed19"
        self.data_go = bytearray([0x01, 0x00])
        self.data_nogo = bytearray([0x00, 0x00])
        
    def prot_close(self):
        """
        Initiates the protocol to close the motor by writing to the CLOSE_MOTOR_HDLE characteristic.
        """
        self.perform_action(self.write_characteristic(self.CLOSE_MOTOR_HDLE, self.data_go))

    def prot_open(self):
        """
        Initiates the protocol to open the motor by writing to the OPEN_MOTOR_HDLE characteristic.
        """
        self.perform_action(self.write_characteristic(self.OPEN_MOTOR_HDLE, self.data_go))

    def prot_stop(self):
        """
        Stops any ongoing motor action by writing to both the OPEN_MOTOR_HDLE and CLOSE_MOTOR_HDLE characteristics.
        """
        self.perform_action(self.write_characteristic(self.CLOSE_MOTOR_HDLE, self.data_nogo))
        self.perform_action(self.write_characteristic(self.OPEN_MOTOR_HDLE, self.data_nogo))


In [3]:
indy = IndyClientThread('BFDC2B32-2AB9-A609-040E-1D1B42D3D68E')
indy.start()
connected = indy.perform_action(indy.connect())
assert(connected == True)
assert(indy.connected == True)

An error occurred: Device with address BFDC2B32-2AB9-A609-040E-1D1B42D3D68E was not found


AssertionError: 

# Simple Movements


In [4]:
indy.prot_close()
time.sleep(2.0)
indy.prot_open()
time.sleep(2.0)

An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet


# Simple movements


In [5]:

indy.prot_open()
time.sleep(0.2)
indy.prot_close()
time.sleep(0.1)
indy.prot_stop()

An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet


# GUI

In [6]:
import ipywidgets as widgets
from IPython.display import display

# Define the generic functions to be connected to the buttons
def left(b):
    indy.prot_close()
    print("Navigating to the far left...")

def step_left(b):
    indy.prot_close()
    time.sleep(0.1)
    indy.prot_stop()
    print("Stepping left...")

def step_right(b):
    indy.prot_open()
    time.sleep(0.1)
    indy.prot_stop()
    print("Stepping right...")

def right(b):
    indy.prot_open()
    print("Navigating to the far right...")

# Create the buttons
button_left = widgets.Button(description="<<")
button_step_left = widgets.Button(description="<")
button_step_right = widgets.Button(description=">")
button_right = widgets.Button(description=">>")

# Connect the buttons to their respective functions
button_left.on_click(left)
button_step_left.on_click(step_left)
button_step_right.on_click(step_right)
button_right.on_click(right)

# Group the buttons in a horizontal box (HBox)
buttons = widgets.HBox([button_left, button_step_left, button_step_right, button_right])

# Display the buttons
display(buttons)

An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet
Stepping left...
An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet
An error occurred: Service Discovery has not been performed yet
Stepping right...
An error occurred: Service Discovery has not been performed yet
Navigating to the far right...
An error occurred: Service Discovery has not been performed yet
Navigating to the far left...
